# Tune a model, and know whether it helped

Tuning is a search, and **the best of N tries looks good partly by luck**. The lab's tuning has three safeguards built in, and this notebook keeps them:

1. The search uses **only earlier seasons**, ranked by an inner walk-forward.
2. The winner is **frozen**, then scored on later seasons that no candidate ever saw.
3. Every contender is compared on the **same games**, with a paired confidence interval for the *difference*. The untuned version stays in the comparison.

The winner's own search score (the "inner" score) is best-of-N and optimistic by construction. **Only the outer numbers count.** For football, 2025 is the repo's
touch-once holdout: the lab never scores it, and neither should a spec.

Runs are lab jobs under hard memory, CPU and time limits, so you can close the notebook while one runs; `nb.wait(job)` just follows it.

In [ ]:
from mllab import nb
nb.specs()

## NFL: a tuned twin of the headline model

First, the settings the repo's model uses today. They are the starting point the search is compared with:

In [ ]:
nb.contenders("nfl_walkforward")[["name", "kind", "family", "params", "seeds"]]

`tune_spec` copies the spec and adds `xgb_pure_tuned`: the same model, with a random search over `space` (the starter space is below; edit it, or pass your own).
`seeds` averages several random seeds, which matters because a single XGBoost fit moves a season by a point or two on its own.

In [ ]:
nb.space("xgboost")        # the starter search space: {param: {low, high, [log], [type: int]}} or {param: [choices]}

In [ ]:
nfl_path = nb.tune_spec("nfl_walkforward", "my_nfl_tune", "xgb_pure", trials=40, seeds=[1, 2, 3])
print(nfl_path)
nb.contenders(nfl_path)[["name", "tuned", "seeds"]]

In [ ]:
nfl_job = nb.submit("backtest", nfl_path)
nb.wait(nfl_job)

### Read it

`vs` is Elo by default. A **negative** difference is better than Elo. The verdict comes from the confidence interval: only *better* and *worse* mean anything, and
*not distinguishable* is the usual honest answer.

In [ ]:
nb.results(nfl_job)

What the search found, and how much of its inner gain is probably luck:

In [ ]:
nb.tuned(nfl_job)

In [ ]:
nb.trials(nfl_job, "xgb_pure_tuned")   # if the top few are all close, the surface is flat and the 'best' is not special

Compare the tuned model with its **own untuned twin**: `nb.results(nfl_job, vs="xgb_pure")`. If that interval spans zero, tuning did not help, and the simpler
default stands. (On the lab's earlier run of this experiment it did not.)

In [ ]:
nb.results(nfl_job, vs="xgb_pure")

## CFB

The same, for FBS. (FCS has its own spec, `cfb_fcs_walkforward`.) College football has fewer seasons of history in the export, so expect wider intervals.

In [ ]:
cfb_path = nb.tune_spec("cfb_fbs_walkforward", "my_cfb_tune", "xgb", trials=30, seeds=[1, 2, 3])
cfb_job = nb.submit("backtest", cfb_path)
nb.wait(cfb_job)

In [ ]:
nb.results(cfb_job, vs="xgb")

In [ ]:
nb.tuned(cfb_job)

## MLB: a small grid of fixed settings

MLB is scored only on 2026 games, so there are no whole seasons to hold out and search over. The honest tool is a **small grid**: each combination is a contender, all
scored on the same games against production (`prod`). `grid_spec` refuses a grid above 12 combinations, because every extra combination adds minutes and adds to the
chance that the winner is luck.

In [ ]:
nb.contenders("mlb_v10_param_sweep")[["name", "kind", "family", "params"]]

In [ ]:
mlb_path = nb.grid_spec("mlb_v10_param_sweep", "my_mlb_grid", "xgb_d2", {"max_depth": [2, 3], "learning_rate": [0.03, 0.05]})
nb.contenders(mlb_path)[["name", "params"]]

In [ ]:
mlb_job = nb.submit("backtest", mlb_path)
nb.wait(mlb_job)

In [ ]:
nb.results(mlb_job, vs="prod")

With about 2,000 games, only differences of two or three accuracy points are detectable, so most MLB grids come back *not distinguishable from prod*. That is a result:
it says the settings are not what limits the model.

## If something does win

* Treat it as a **candidate**. Confirm it on games none of the contenders has seen (for MLB, wait for more games and re-run the backtest).
* Then train it properly and register it in `03_train_and_register`, and point a channel at it in the control app.
* Do not tune on the football 2025 holdout, and do not re-run a search until a result you like appears: that is the same luck, taken again.